# 02_ex_agreement_topic — exploration and proposal

Analyst/data scientist notebook for source exploration, exploratory transforms, and proposal evidence.

> AI output in this notebook is advisory and requires human validation.
> Governance approval belongs to `01_data_sharing_agreement`; production enforcement belongs to `03_pc`.


## 01 Introduction and scope

Document:
- `agreement_id`
- `topic`
- source being explored
- question being answered
- approved usage context from the data sharing agreement


In [ ]:
topic = "REPLACE_TOPIC"
table_name = "REPLACE_TABLE_NAME_OR_ALIAS"


## 02 Agreement context

Capture approved context from `01_data_sharing_agreement_<agreement>`:
- approved usage scope
- stakeholder owner/steward
- known restrictions relevant to this exploration


## 03 Configuration and setup


In [ ]:
%run 00_env_config


## 04 Metadata-first inspection

Before profiling the source again, check what is already known. Existing metadata may already contain profile evidence, approved DQ rules, sensitivity labels, business context, and prior proposal evidence.


In [ ]:
from fabricops_kit import get_path, lakehouse_table_read, lakehouse_csv_read, lakehouse_parquet_read_as_spark, warehouse_read, profile_dataframe, load_approved_dq_rules, draft_dq_rules, write_dq_rules

metadata_path = get_path(ENV_NAME, "metadata", config=FABRIC_CONFIG)
dq_metadata_table = FABRIC_CONFIG.review_workflow_config.dq_approved_table

# Optional metadata-first inspection
# existing_dq_df = lakehouse_table_read(metadata_path, dq_metadata_table)
# approved_active_rules = load_approved_dq_rules(existing_dq_df, table_name=table_name)
# existing_governance_df = lakehouse_table_read(metadata_path, FABRIC_CONFIG.review_workflow_config.governance_approved_table)
# existing_business_context_df = lakehouse_table_read(metadata_path, FABRIC_CONFIG.review_workflow_config.business_context_approved_table)


## 06 Data ingestion for exploration

Choose one source loading pattern only when a fresh read is needed.


In [ ]:
# Option A: Lakehouse Delta table
source_df = lakehouse_table_read(
    get_path(ENV_NAME, "source", config=FABRIC_CONFIG),
    table_name,
)

# Option B: Warehouse table
# source_df = warehouse_read(
#     env=ENV_NAME,
#     target="warehouse",
#     schema="dbo",
#     table=table_name,
#     config=FABRIC_CONFIG,
# )

profile_df = profile_dataframe(source_df, table_name=table_name)
profile_df.show(50, truncate=False)


## 07 Source profiling (optional refresh)

Use profiling when existing metadata is missing, stale, or insufficient for the current question.


In [ ]:
# Profiling is executed in the ingestion cell above to keep one profiling pass.
# profile_df.write.mode("append").saveAsTable("REPLACE_PROFILE_OUTPUT_TABLE")


## 08 Exploratory analysis

Inspect distributions, nulls, duplicates, sample records, and capture analyst observations.


In [ ]:
# Example exploration snippets
source_df.selectExpr("count(*) as total_rows").show()
source_df.limit(20).show(truncate=False)

# TODO: add focused analyst observations for this topic.


## 09 AI-assisted DQ rule review (optional)

Optional analyst/engineer-owned DQ flow: load existing approved rules, draft AI candidates when needed, review in the existing DQ widget, persist approved rules, review deactivations, and persist deactivation metadata.


In [ ]:
# 1) Optional: load existing approved/active DQ rules for this table from metadata
dq_metadata_table = FABRIC_CONFIG.review_workflow_config.dq_approved_table
# existing_dq_df = lakehouse_table_read(metadata_path, dq_metadata_table)
# approved_active_rules = load_approved_dq_rules(existing_dq_df, table_name=table_name)

# 2) Optional: generate AI candidate rules only when needed
# profile_for_ai_df = profile_df
# candidate_rules = draft_dq_rules(
#     profile_df=profile_for_ai_df,
#     table_name=table_name,
#     prompt_template=FABRIC_CONFIG.ai_prompt_config.dq_rule_candidate_template,
# )

# 3) Optional: write approved rules to metadata
# write_dq_rules(candidate_rules, metadata_path=metadata_path, table_name=table_name)

# 4) Optional: persist rule deactivations
# deactivation_df.write.mode("append").saveAsTable(dq_metadata_table)


## 10 Findings and proposed metadata updates


In [ ]:
proposal_rows = [
    {
        "agreement_id": agreement_id,
        "topic": topic,
        "table_name": table_name,
        "proposal_type": "dq_candidate",
        "rationale": "REPLACE",
        "source_evidence": "existing metadata / fresh profile / analyst observation",
    },
    {
        "agreement_id": agreement_id,
        "topic": topic,
        "table_name": table_name,
        "proposal_type": "classification_candidate",
        "rationale": "REPLACE",
        "source_evidence": "existing metadata / fresh profile / analyst observation",
    },
]

proposal_df = spark.createDataFrame(proposal_rows)
proposal_df.show(truncate=False)

# proposal_df.write.mode("append").saveAsTable("REPLACE_PROPOSAL_EVIDENCE_TABLE")


## 11 Handoff to governance and pipeline contract

- Governance updates move to `01_data_sharing_agreement_<agreement>` for approval.
- Approved DQ rules are consumed by `03_pc_<agreement>_<pipeline>`; governance/classification updates follow the data agreement workflow.
- Production transforms and enforcement happen in `03_pc`.


## 12 Frozen diagnostics / appendix

Keep one-time diagnostics and deep debug outputs here. Avoid turning exploratory cells into scheduled production behavior in `02_ex`.
